# Harvesting MLOps SharingHub

This notebook demonstrates the harvesting of the SharingHub STAC API with the Resource Registration BB Harvester's STAC workflow.
The source STAC catalog is https://develop.eoepca.org/sharinghub/api/stac/ which will be harvested into the Resource Catalog's STAC API at https://resource-catalogue.develop.eoepca.org/stac.

In [ ]:
import requests
from requests import Session
from requests.auth import HTTPBasicAuth
from pystac_client import Client
import json
from urllib3.exceptions import InsecureRequestWarning

sharinghub_stac_api_url = "https://develop.eoepca.org/sharinghub/api/stac/"
destination_stac_api_url = "https://resource-catalogue.develop.eoepca.org/stac"
#destination_stac_api_url = "https://eoapi.develop.eoepca.org/stac/"

# Suppress the warnings from urllib3
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

# Setup connection to engine API
engine_base_url = "https://registration-harvester-bpm-engine.develop.eoepca.org/engine-rest"
engine_rest_user = "eoepca"
engine_rest_pw = "eoepca"

engine_session = Session()
engine_session.verify = False
engine_session.auth = HTTPBasicAuth(engine_rest_user, engine_rest_pw)

stac_process_key = None
url = f"{engine_base_url}/process-definition"
print(f"GET {url}")
response = engine_session.get(url)
processes = response.json()
if len(processes) == 0:
    print("No workflow definitions")
else:
    for idx, process in enumerate(processes, 1):
        print("%-2s %-28s version: %-5s key: %s" % (idx, process['name'], process['version'], process['key']))
        if process["name"] == "Harvest STAC Catalog":
            stac_process_key = process["key"]

## Explore SharingHub STAC API

List collections, items and their assets contained in SharingHub. It can also be explored by STAC Browser: https://develop.eoepca.org/sharinghub/#/

In [ ]:
cat = Client.open(sharinghub_stac_api_url)
collections = cat.get_all_collections()
for collection in collections:
    print(f"Collection: {collection.id}")
    items = collection.get_items()
    for item in items:
        print(f"  ⊢ Item: {item.id}")
        assets = item.get_assets()
        for name, asset in assets.items():
            print(f"     ⊢ Asset: {name}")

## Execute STAC workflow of Harvester

### (1) Define workflow inputs

In [ ]:
variables = {
    "stac_catalog_source" : {
        "value": sharinghub_stac_api_url
    },
    "stac_update_collections": {
        "value": True,
    },
    "stac_api_destination_url": {
        "value": destination_stac_api_url
    },
    "datetime": {
        "value": "2024-01-01T00:00:00/2026-01-01T00:00:00"
    },    
}
print(json.dumps(variables, indent=2))

### (2) Start the workflow

In [ ]:
# Send HTTP request to start the workflow
body = {}
body["variables"] = variables
response = engine_session.post(url=f"{engine_base_url}/process-definition/key/{stac_process_key}/start", json=body)
print(response.status_code)
print(f"Created process instance {response.json()["id"]}")

## Explore Resource Catalog STAC API

Resource Catalog STAC API can also explored by STAC Browser: https://radiantearth.github.io/stac-browser/#/external/resource-catalogue.develop.eoepca.org/stac?.language=en

In [ ]:
cat = Client.open(destination_stac_api_url)
collections = cat.get_all_collections()
for collection in collections:
    if collection.id in ["ai-model", "dataset"]:
        print(f"Collection: {collection.id}")
        items = collection.get_items()
        for item in items:
            print(f"  ⊢ Item: {item.id}")
            assets = item.get_assets()
            for name, asset in assets.items():
                print(f"     ⊢ Asset: {name}")